# Batch generation pilot

Tests batch sizes 1, 4, 8 on 20 prompts. This measures corpus generation, not similarity search.

In [ ]:
%pip -q install 'transformers>=4.44,<5' accelerate safetensors


In [ ]:
import json, time, statistics, platform
from pathlib import Path
from datetime import datetime, timezone
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
torch.manual_seed(42)
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto").eval()
prompts = [
"Explain transformer attention.","Describe phishing prevention.","How does inflation affect savings?",
"Explain electric vehicle batteries.","What is supervised learning?",
"What causes earthquakes?","Explain solar power.","How do database indexes work?",
"Describe how vaccines work.","How do movie recommenders work?",
"Explain computer network routing.","What is a SQL transaction?",
"Describe the water cycle.","How does cloud computing work?",
"Explain carbon emissions.","What is a neural network?",
"Describe password hashing.","How does GPS positioning work?",
"Explain photosynthesis.","What is an operating system scheduler?"
]
def render(p):
    return tok.apply_chat_template([
        {"role":"system","content":"Answer in English, clearly and concisely, in about 65-80 words. Use complete sentences."},
        {"role":"user","content":p}
    ], tokenize=False, add_generation_prompt=True)

def run(batch_size):
    rows = []
    start = time.perf_counter()
    for offset in range(0,len(prompts),batch_size):
        group=prompts[offset:offset+batch_size]
        enc=tok([render(p) for p in group],return_tensors="pt",padding=True).to(model.device)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0=time.perf_counter()
        with torch.inference_mode():
            out=model.generate(**enc,max_new_tokens=100,do_sample=False,pad_token_id=tok.eos_token_id)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        for p, seq in zip(group,out):
            continuation=seq[enc.input_ids.shape[1]:]
            answer=tok.decode(continuation,skip_special_tokens=True).strip()
            tokens=len(tok.encode(answer,add_special_tokens=False))
            rows.append({"prompt":p,"response":answer,"response_tokens":tokens,
                         "batch_size":batch_size,"batch_generation_seconds":round(elapsed,4)})
    return rows,time.perf_counter()-start

# Warm up GPU before timing both strategies.
_ = run(1) if False else None
warm=tok([render(prompts[0])],return_tensors="pt").to(model.device)
with torch.inference_mode():
    model.generate(**warm,max_new_tokens=16,do_sample=False,pad_token_id=tok.eos_token_id)

results={}
for bs in [1,4,8]:
    rows,secs=run(bs)
    lengths=[r["response_tokens"] for r in rows]
    results[str(bs)]={"seconds":secs,"docs_per_second":len(rows)/secs,
        "mean_tokens":statistics.mean(lengths),"min_tokens":min(lengths),
        "max_tokens":max(lengths),"empty_count":sum(not r["response"] for r in rows),
        "hit_100_token_cap":sum(t>=99 for t in lengths),"rows":rows}
    print(f"batch={bs}: {secs:.1f}s, {len(rows)/secs:.2f} docs/s, "
          f"avg={statistics.mean(lengths):.1f} tokens, min={min(lengths)}, "
          f"at cap={sum(t>=99 for t in lengths)}/20")
fastest=max(results,key=lambda b:results[b]["docs_per_second"])
rate=results[fastest]["docs_per_second"]
print("Fastest batch size:",fastest)
for n in [5000,20000]:
    print(f"Projected {n:,} docs: {n/rate/3600:.2f} hours (extrapolation only)")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder=Path('/content/drive/MyDrive/HP-SS-Engine/data/raw')
folder.mkdir(parents=True,exist_ok=True)
for bs,item in results.items():
    with (folder/f'qwen_pilot_batch_{bs}.jsonl').open('w',encoding='utf-8') as f:
        for row in item['rows']:
            f.write(json.dumps(row,ensure_ascii=False)+'\n')
summary={"timestamp_utc":datetime.now(timezone.utc).isoformat(),
         "model":MODEL,"device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
         "n_prompts":len(prompts),"max_new_tokens":100,"results":{
             k:{x:y for x,y in v.items() if x!="rows"} for k,v in results.items()}}
(folder/'qwen_pilot_batch_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print("Saved to:",folder)
